# 策略效果瓶颈：信号误差归因与固定风险预算改进

问题不是“再加模型”：已保存的财报特征 IC 上升但 MSE/费用事件恶化；旧 SOXL 规则存在超额，但回撤与尾部损失显著超出 SOXX。先复核冻结证据，再仅测试一项结构改进：用前一收盘已知的 60 日 SOXX/SOXL 波动率比例限制 SOXL 目标仓位。

`plan.json` 在任何新候选回放前固定；原规则、信号、成本、历史窗口与晋升门不改。新策略使用独立影子旧账本的既有意图，自身仓位不反向影响影子信号。相同目标调仓方式的无上限对照用于隔离风险上限效果。该设计可能牺牲上涨收益，也不能保证路径风险不超过 SOXX。没有参数搜索、收益保证或生产切换。

In [1]:
import os, sys, json, hashlib
from pathlib import Path
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "SPEC.md").exists())
EVIDENCE_ROOT = Path(os.environ.get("SOXL_CANONICAL_EVIDENCE_ROOT", str(ROOT))).resolve()
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R
from src.data.legacy_execution_view import load_legacy_execution_view
from src.backtest.legacy_risk_budget import build_legacy_risk_targets, run_native_target_replay
from src.analysis.strategy_bottlenecks import analyze_forecast_increment
from src.workflow.legacy_soxl_repair import path_metrics, bootstrap_excess
NAME = "strategy_bottleneck_risk_budget_20261008"
CARD = ROOT / "research" / NAME
sha = lambda path: hashlib.sha256(Path(path).read_bytes()).hexdigest()
plan = json.loads((CARD / "plan.json").read_text())
binding = json.loads((CARD / "run_binding.json").read_text())
assert binding["plan_sha256"] == sha(CARD / "plan.json")
notebook = json.loads((CARD / (NAME + ".ipynb")).read_text())
code_text = "\n".join("".join(c["source"]) for c in notebook["cells"] if c["cell_type"] == "code")
assert hashlib.sha256(code_text.encode()).hexdigest() == binding["notebook_code_sha256"]
assert all(sha(ROOT / path) == digest for path, digest in binding["source_sha256"].items())
assert binding["focused_tests_passed"] and plan["replay_budget"] == 8 and plan["model_fit_budget"] == 0
provider, provider_audit = load_legacy_execution_view(ROOT)
qlib.init(provider_uri=str(provider), region="us", kernels=1)
calendar = pd.DatetimeIndex(D.calendar(freq="day", future=False))
raw = D.features(["SOXL", "SOXX"], ["$open", "$close"], start_time=calendar[0], end_time=calendar[-1], freq="day")
closes = pd.DataFrame({symbol:raw.xs(symbol, level="instrument")["$close"].reindex(calendar).astype(float) for symbol in ("SOXL","SOXX")})
market = pd.DataFrame({"open":raw.xs("SOXL",level="instrument")["$open"].reindex(calendar).astype(float), "close":closes.SOXL})
assert np.isfinite(market.to_numpy()).all() and np.isfinite(closes.to_numpy()).all()
source_bindings = {}
def load_frozen(experiment, names):
    card = EVIDENCE_ROOT / "research" / experiment
    summary = json.loads((card / "summary.json").read_text())
    assert sha(card / "summary.json") == plan["source_summary_sha256"][experiment]
    with R.uri_context(str(card / "qlib_recorder")):
        rec = R.get_recorder(experiment_name=experiment, recorder_id=summary["recorder_id"])
        assert rec.client.get_run(rec.id).info.status == "FINISHED"
        artifacts = Path(rec.get_local_dir()) / "artifacts"
        hashes = summary.get("recorder_artifact_sha256")
        if hashes is None:
            ref = json.loads((card / "recorder_reference.json").read_text())
            hashes = {key: (value["sha256"] if isinstance(value,dict) else value) for key,value in ref["artifact_sha256"].items()}
        bound = {}
        result = {}
        for name in names:
            assert sha(artifacts/name) == hashes[name], name
            result[name] = rec.load_object(name)
            assert sha(artifacts/name) == hashes[name], name
            bound[name] = hashes[name]
        source_bindings[experiment] = {"recorder_id": rec.id, "artifacts":bound}
    return summary, result
legacy_summary, legacy_objects = load_frozen("legacy_soxl_repair_20261005", ["analysis/ledgers.pkl", "portfolio_analysis/native_reports.pkl"])
earnings_summary, earnings_objects = load_frozen("earnings_enhancement_20261005", ["analysis/predictions.pkl"])
ledgers = legacy_objects["analysis/ledgers.pkl"]
parent_paths = legacy_objects["portfolio_analysis/native_reports.pkl"]
assert provider_audit["native_frame_sha256"] == legacy_summary["source_audit"]["execution"]["native_frame_sha256"]
print({"provider":plan["execution_provider_uri"],"as_of":plan["as_of"],"canonical_sessions":len(calendar),"source_recorders":source_bindings})

2026/10/08 10:57:52 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[3010:MainThread](2026-10-08 10:57:52,776) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[3010:MainThread](2026-10-08 10:57:52,778) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[3010:MainThread](2026-10-08 10:57:52,778) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace-analysis-opt/data/qlib/published_futu_affine_20260925_v1')}


...

[3010:MainThread](2026-10-08 10:57:52,794) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace-analysis-opt/data/qlib/legacy_soxl_execution_20260925_v1')}


[3010:MainThread](2026-10-08 10:57:52,813) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[3010:MainThread](2026-10-08 10:57:52,813) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[3010:MainThread](2026-10-08 10:57:52,814) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace-analysis-opt/data/qlib/legacy_soxl_execution_20260925_v1')}


{'provider': 'data/qlib/legacy_soxl_execution_20260925_v1', 'as_of': '2026-09-25', 'canonical_sessions': 4162, 'source_recorders': {'legacy_soxl_repair_20261005': {'recorder_id': 'a899b1df50274e769ec1d7fead8676ed', 'artifacts': {'analysis/ledgers.pkl': '3b3d168f25de447dc7ca0f8883d877306a41522a02d69a31404edbbd4ea88f2c', 'portfolio_analysis/native_reports.pkl': '1f3f441cc5582bd3faa79cb20dd1233bd6d2129e1eefb545cd4861c5fcbb4a24'}}, 'earnings_enhancement_20261005': {'recorder_id': 'b3674e7f2d7c41a5b8827c99f316969a', 'artifacts': {'analysis/predictions.pkl': 'f97177e6fdd4f250fd2b3f6e0b464a853b163931d73427019d9adbb9566363f8'}}}}


## 信号为什么失效

令增量预测 δ=full−base，则 MSE(full)−MSE(base)=E[δ²]−2E[δ(y−base)]。第一项是新增预测幅度，第二项度量增量是否修正原有错误。此处只归因既有预测，不据结果反号、删年份、重新校准或筛选阈值。费用事件为重叠孤立 H5 事件，不能复利/年化为组合收益。

In [2]:
forecast = analyze_forecast_increment(earnings_objects["analysis/predictions.pkl"])
forecast["mse_attribution"]

,head,slice,rows,first_signal,last_signal,base_mse,full_mse,extra_forecast_energy,twice_residual_alignment,full_minus_base_MSE,...,identity_within_roundoff,increment_residual_correlation,increment_residual_correlation_undefined_reason,base_ic,base_ic_undefined_reason,full_ic,full_ic_undefined_reason,delta_ic_full_minus_base,SSE_difference_fraction_of_pooled,SSE_fraction_undefined_reason
0,cash5,pooled,1762,2016-01-04T00:00:00,2022-12-30T00:00:00,0.001652,0.001756,0.000058,-0.000047,0.000105,...,True,-0.078195,None,0.042432,None,0.069832,None,0.027400,1.000000,None
1,cash5,2016,252,2016-01-04T00:00:00,2016-12-30T00:00:00,0.000931,0.000958,0.000013,-0.000015,0.000027,...,True,0.241046,None,0.253790,None,0.307703,None,0.053913,0.037389,None
2,cash5,2017,251,2017-01-03T00:00:00,2017-12-29T00:00:00,0.000621,0.000636,0.000025,0.000010,0.000015,...,True,0.019739,None,0.096976,None,0.080579,None,-0.016397,0.020386,None
3,cash5,2018,251,2018-01-02T00:00:00,2018-12-31T00:00:00,0.001342,0.001333,0.000020,0.000029,-0.000009,...,True,-0.075291,None,0.174002,None,0.115570,None,-0.058431,-0.012204,None
4,cash5,2019,252,2019-01-02T00:00:00,2019-12-31T00:00:00,0.001153,0.001147,0.000009,0.000015,-0.000006,...,True,0.005120,None,0.148607,None,0.172794,None,0.024187,-0.008328,None
5,cash5,2020,253,2020-01-02T00:00:00,2020-12-31T00:00:00,0.002743,0.002721,0.000009,0.000031,-0.000022,...,True,0.105211,None,0.195588,None,0.224863,None,0.029275,-0.030163,None
6,cash5,2021,252,2021-01-04T00:00:00,2021-12-31T00:00:00,0.001288,0.001376,0.000082,-0.000007,0.000088,...,True,0.006425,None,0.078439,None,0.102780,None,0.024342,0.120692,None
7,cash5,2022,251,2022-01-03T00:00:00,2022-12-30T00:00:00,0.003481,0.004123,0.000249,-0.000393,0.000642,...,True,-0.090595,None,0.053111,None,-0.040892,None,-0.094003,0.872229,None
8,leverage5,pooled,1762,2016-01-04T00:00:00,2022-12-30T00:00:00,0.007369,0.007845,0.000270,-0.000205,0.000476,...,True,-0.077277,None,0.009434,None,0.049621,None,0.040187,1.000000,None
9,leverage5,2016,252,2016-01-04T00:00:00,2016-12-30T00:00:00,0.003898,0.004021,0.000057,-0.000066,0.000123,...,True,0.235507,None,0.226053,None,0.286516,None,0.060463,0.036983,None


In [3]:
forecast["event_calibration"]

,head,slice,rows,first_signal,last_signal,family,cost_bps,events,coverage,hurdle_bp,forecast_mean_net_bp,realized_mean_net_bp,overestimation_bp,positive_event_fraction,conditional_mean_undefined_reason,event_scope
0,cash5,pooled,1762,2016-01-04T00:00:00,2022-12-30T00:00:00,base,10.0,204,0.115778,20.000007,21.710744,-7.387057,29.097801,0.426471,None,isolated_overlapping_H5_SOXX_roundtrip_not_por...
1,cash5,pooled,1762,2016-01-04T00:00:00,2022-12-30T00:00:00,base_plus_earnings,10.0,451,0.255959,20.000007,43.154920,-29.026825,72.181745,0.441242,None,isolated_overlapping_H5_SOXX_roundtrip_not_por...
2,cash5,pooled,1762,2016-01-04T00:00:00,2022-12-30T00:00:00,base,25.0,58,0.032917,50.000104,15.734693,-36.773948,52.508642,0.327586,None,isolated_overlapping_H5_SOXX_roundtrip_not_por...
3,cash5,pooled,1762,2016-01-04T00:00:00,2022-12-30T00:00:00,base_plus_earnings,25.0,238,0.135074,50.000104,39.903354,-45.388610,85.291964,0.415966,None,isolated_overlapping_H5_SOXX_roundtrip_not_por...
4,cash5,2016,252,2016-01-04T00:00:00,2016-12-30T00:00:00,base,10.0,59,0.234127,20.000007,23.040468,33.784532,-10.744064,0.457627,None,isolated_overlapping_H5_SOXX_roundtrip_not_por...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,leverage5,2021,252,2021-01-04T00:00:00,2021-12-31T00:00:00,base_plus_earnings,25.0,1,0.003968,100.000208,5.771617,-1305.986649,1311.758265,0.000000,None,isolated_overlapping_H5_SOXX_roundtrip_not_por...
60,leverage5,2022,251,2022-01-03T00:00:00,2022-12-30T00:00:00,base,10.0,239,0.952191,40.000013,63.208240,-365.658217,428.866456,0.389121,None,isolated_overlapping_H5_SOXX_roundtrip_not_por...
61,leverage5,2022,251,2022-01-03T00:00:00,2022-12-30T00:00:00,base_plus_earnings,10.0,190,0.756972,40.000013,303.031439,-358.017988,661.049427,0.389474,None,isolated_overlapping_H5_SOXX_roundtrip_not_por...
62,leverage5,2022,251,2022-01-03T00:00:00,2022-12-30T00:00:00,base,25.0,121,0.482072,100.000208,30.002540,-321.642289,351.644829,0.363636,None,isolated_overlapping_H5_SOXX_roundtrip_not_por...


## 风险改进的事前意图

每种成本使用同成本影子账本：前一收盘的原暴露，加原规则当时发出的净买卖意图。候选与无上限对照都在下一开盘按目标仓位调仓；两者与原 LIFO/整数股规则并非同一执行策略。60 日窗口事前固定，不按结果更换。保留旧规则的现金状态，因此本项不承诺修复近期现金启动空仓。

In [4]:
targets = {}
for book, bounds in plan["books"].items():
    for cost in plan["costs_bps"]:
        legacy = ledgers[(book,cost)]
        original = legacy.daily.set_index("date")
        native = parent_paths[(book,cost,"legacy_repaired")]["daily"]
        np.testing.assert_allclose(original.exposure.to_numpy(),native.exposure.to_numpy(),atol=1e-12,rtol=1e-12)
        np.testing.assert_array_equal(original.close.to_numpy(),market.loc[original.index,"close"].to_numpy())
        target = build_legacy_risk_targets(legacy, closes, volatility_window=plan["volatility_window"])
        assert target.index.equals(native.index)
        targets[(book,cost)] = target
print({str(key):{"rows":len(value), "capped_sessions":int((value.capped_weight < value.raw_weight).sum()),
                  "maximum_cap":float(value.risk_cap.max())} for key,value in targets.items()})

{"('continuous', 10)": {'rows': 2698, 'capped_sessions': 1674, 'maximum_cap': 0.374629985766515}, "('continuous', 25)": {'rows': 2698, 'capped_sessions': 1674, 'maximum_cap': 0.374629985766515}, "('recent_cash_start', 10)": {'rows': 39, 'capped_sessions': 0, 'maximum_cap': 0.346479732349579}, "('recent_cash_start', 25)": {'rows': 39, 'capped_sessions': 0, 'maximum_cap': 0.346479732349579}}


## 八条原生 Qlib 路径与独立记账

同一初始现金、每侧 10/25bp、连续持仓不按年度重置。原生 Backtest/Exchange/Account/Position 的报告为主证据；独立现金/股数/费用账本逐日对账。单次登记回放，完成后重开 notebook 读取已完成记录，不重新筛选策略。

In [5]:
uri = str((CARD / "qlib_recorder").resolve())
attempt = CARD / "attempt.json"
if attempt.exists():
    completed = json.loads(attempt.read_text())
    assert completed["status"] == "complete" and completed["binding_sha256"] == sha(CARD / "run_binding.json"), "blocked: incomplete or changed registered attempt"
    with R.uri_context(uri):
        rec = R.get_recorder(experiment_name=NAME,recorder_id=completed["recorder_id"])
        assert rec.client.get_run(rec.id).info.status == "FINISHED"
        artifact_root = Path(rec.get_local_dir()) / "artifacts"
        assert all(sha(artifact_root / name) == digest for name,digest in completed["artifact_sha256"].items())
        new_paths = rec.load_object("portfolio_analysis/native_reports.pkl")
        reconciliations = rec.load_object("analysis/reconciliation.pkl")
else:
    attempt.write_text(json.dumps({"status":"started","binding_sha256":sha(CARD / "run_binding.json")})+"\n")
    new_paths, reconciliations, independent_ledgers = {}, {}, {}
    with R.start(experiment_name=NAME, uri=uri):
        rec = R.get_recorder()
        R.log_params(plan_sha256=sha(CARD / "plan.json"), binding_sha256=sha(CARD / "run_binding.json"), source_recorder_id=plan["source_recorder_id"], volatility_window=60)
        R.save_objects(artifact_path="inputs", **{"target_weights.pkl": targets, "source_bindings.pkl":source_bindings})
        for book, bounds in plan["books"].items():
            for cost in plan["costs_bps"]:
                for variant, column in (("raw_intended_weight_control","raw_weight"),("volatility_capped","capped_weight")):
                    result = run_native_target_replay(market, targets[(book,cost)], weight_column=column, cost_bps=cost, initial_cash=plan["initial_cash"])
                    new_paths[(book,cost,variant)] = result["native"]
                    reconciliations[(book,cost,variant)] = result["audit"]
                    independent_ledgers[(book,cost,variant)] = result["engine"]
                    print(book,cost,variant,result["audit"]["status"])
        assert len(new_paths) == plan["replay_budget"] and all(x["status"] == "passed" for x in reconciliations.values())
        R.save_objects(artifact_path="portfolio_analysis", **{"native_reports.pkl":new_paths})
        R.save_objects(artifact_path="analysis", **{"reconciliation.pkl":reconciliations,"independent_ledgers.pkl":independent_ledgers})
        for path in [CARD / "plan.json", CARD / "run_binding.json", CARD / "workflow.yaml"]:
            rec.log_artifact(str(path),artifact_path="card")
    completed={"status":"complete","binding_sha256":sha(CARD / "run_binding.json"),"recorder_id":rec.id,"experiment_id":rec.experiment_id,"replays":len(new_paths)}
    artifact_root = Path(rec.get_local_dir()) / "artifacts"
    completed["artifact_sha256"] = {name:sha(artifact_root/name) for name in ("portfolio_analysis/native_reports.pkl","analysis/reconciliation.pkl","analysis/independent_ledgers.pkl","inputs/target_weights.pkl")}
    attempt.write_text(json.dumps(completed,indent=2)+"\n")
recorder_id=completed["recorder_id"]
print({"recorder_id":recorder_id,"native_reconciliation":"passed","new_paths":len(new_paths)})

[3010:MainThread](2026-10-08 10:57:54,917) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name strategy_bottleneck_risk_budget_20261008.


[3010:MainThread](2026-10-08 10:57:54,918) INFO - qlib.workflow - [exp.py:258] - Experiment 216598613405890605 starts running ...


[3010:MainThread](2026-10-08 10:57:55,007) INFO - qlib.workflow - [recorder.py:345] - Recorder 42a5f95d21c3413bac125fa5b4f1085a starts running under Experiment 216598613405890605 ...


[3010:MainThread](2026-10-08 10:57:55,129) INFO - qlib.backtest caller - [__init__.py:93] - Create new exchange


[3010:MainThread](2026-10-08 10:57:55,135) WARNING - qlib.online operator - [exchange.py:226] - factor.day.bin file not exists or factor contains `nan`. Order using adjusted_price.


...

backtest loop:   0%|          | 0/39 [00:00<?, ?it/s]

recent_cash_start 25 volatility_capped passed


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)


[3010:MainThread](2026-10-08 10:58:03,881) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


{'recorder_id': '42a5f95d21c3413bac125fa5b4f1085a', 'native_reconciliation': 'passed', 'new_paths': 8}


## 收益、回撤与门禁

主候选固定为 volatility_capped，不从对照方案中事后挑赢家。开发期沿用 6/7 正超额年份、SOXX 回撤/CVaR 上限及两种区块 bootstrap 门；复用期只是诊断。历史风险改进与稳定超额资格分别报告。

In [6]:
comparison, annual, bootstrap, gates, draws = [], [], [], [], {}
variants=("legacy_repaired","raw_intended_weight_control","volatility_capped","soxx_buy_hold","soxl_fixed_50")
paths={**parent_paths,**new_paths}
for window,spec in plan["windows"].items():
    for cost in plan["costs_bps"]:
        bench=paths[(spec["book"],cost,"soxx_buy_hold")]["daily"].loc[spec["start"]:spec["end"]]
        for variant in variants:
            daily=paths[(spec["book"],cost,variant)]["daily"].loc[spec["start"]:spec["end"]]
            assert daily.index.equals(bench.index)
            excess=np.log1p(daily.net_return)-np.log1p(bench.net_return)
            comparison.append({"window":window,"cost_bps":cost,"variant":variant,**path_metrics(daily),
                "log_excess_vs_soxx":float(excess.sum()),"relative_wealth_vs_soxx":float(np.exp(excess.sum()))})
            for year in sorted(daily.index.year.unique()):
                selected=daily.index.year==year
                annual.append({"window":window,"cost_bps":cost,"variant":variant,"year":int(year),**path_metrics(daily.loc[selected]),"log_excess_vs_soxx":float(excess.loc[selected].sum())})
        if window != "recent_cash_start":
            candidate=paths[(spec["book"],cost,"volatility_capped")]["daily"].loc[spec["start"]:spec["end"]]
            cm,bm=path_metrics(candidate),path_metrics(bench)
            excess=np.log1p(candidate.net_return)-np.log1p(bench.net_return)
            per_year=excess.groupby(excess.index.year).sum()
            checks={"positive_net_log_excess":bool(excess.sum()>0),"max_drawdown_not_worse_soxx":cm["max_drawdown_pct"]>=bm["max_drawdown_pct"]-1e-10,
                    "cvar5_not_worse_soxx":cm["cvar5_pct"]>=bm["cvar5_pct"]-1e-10}
            if window=="development":
                checks["at_least_6_of_7_positive_years"]=bool(len(per_year)==7 and (per_year>0).sum()>=6)
                checks["median_annual_log_excess_positive"]=bool(per_year.median()>0)
            for block in plan["bootstrap"]["blocks"]:
                result=bootstrap_excess(candidate,bench,mean_block=block,resamples=plan["bootstrap"]["resamples"],seed=plan["bootstrap"]["seed"])
                draws[(window,cost,block)]=result.pop("_draws")
                bootstrap.append({"window":window,"cost_bps":cost,**result})
                checks[f"block_{block}_positive_lower_bound"]=result["lower_bound_positive"]
            gates.append({"window":window,"cost_bps":cost,"positive_years":int((per_year>0).sum()),**checks,"all_pass":bool(all(checks.values()))})
comparison=pd.DataFrame(comparison);annual=pd.DataFrame(annual);bootstrap=pd.DataFrame(bootstrap);gates=pd.DataFrame(gates)
comparison.loc[comparison.cost_bps.eq(25),["window","variant","cagr_pct","max_drawdown_pct","cvar5_pct","mean_exposure_pct","turnover","relative_wealth_vs_soxx"]]

,window,variant,cagr_pct,max_drawdown_pct,cvar5_pct,mean_exposure_pct,turnover,relative_wealth_vs_soxx
5,development,legacy_repaired,23.700771,-70.066725,-7.529795,36.870220,5.161449,1.039386
6,development,raw_intended_weight_control,23.196103,-70.137750,-7.597283,36.852438,15.012494,1.010097
7,development,volatility_capped,14.856954,-47.232633,-4.485863,24.178606,14.661532,0.618767
8,development,soxx_buy_hold,23.019224,-45.751346,-4.742697,99.989326,0.997266,1.000000
9,development,soxl_fixed_50,25.892177,-63.515897,-7.187256,49.987183,19.162885,1.175170
15,reused_diagnostic,legacy_repaired,63.470444,-59.731294,-7.103970,49.426062,3.020187,1.226453
16,reused_diagnostic,raw_intended_weight_control,62.193876,-60.698361,-7.209918,49.362330,12.122739,1.191255
17,reused_diagnostic,volatility_capped,38.005050,-44.504097,-4.989895,31.997621,10.753551,0.653863
18,reused_diagnostic,soxx_buy_hold,54.728986,-41.358845,-5.398865,99.997051,0.000000,1.000000
19,reused_diagnostic,soxl_fixed_50,68.483366,-59.848803,-8.134259,50.002981,12.181786,1.372063


In [7]:
gates

,window,cost_bps,positive_years,positive_net_log_excess,max_drawdown_not_worse_soxx,cvar5_not_worse_soxx,at_least_6_of_7_positive_years,median_annual_log_excess_positive,block_20_positive_lower_bound,block_60_positive_lower_bound,all_pass
0,development,10,0,False,False,True,False,False,False,False,False
1,development,25,0,False,False,True,False,False,False,False,False
2,reused_diagnostic,10,0,False,False,True,NaN,NaN,False,False,False
3,reused_diagnostic,25,0,False,False,True,NaN,NaN,False,False,False


## 结论归档

仅保存小表在卡片根目录；目标权重、原生路径、持仓、逐次 bootstrap 均保存在新 Qlib Recorder。原研究结论与 seal 保持原样。

In [8]:
tables={"backtest_benchmark_comparison.csv":comparison,"annual_metrics.csv":annual,"bootstrap.csv":bootstrap,"development_gates.csv":gates,
        "forecast_mse_attribution.csv":forecast["mse_attribution"],"forecast_event_calibration.csv":forecast["event_calibration"]}
for name,table in tables.items(): table.to_csv(CARD/name,index=False)
summary={"experiment":NAME,"record_type":"strategy_bottleneck_structural_risk_diagnostic","provider_uri":plan["execution_provider_uri"],
    "source_provider_uri":plan["source_provider_uri"],"region":"us","market":"all","as_of":plan["as_of"],
    "recorder_id":recorder_id,"experiment_id":completed["experiment_id"],"recorder_uri":str((CARD/"qlib_recorder").relative_to(ROOT)),
    "recorder_relative_dir":str(Path(completed["experiment_id"])/recorder_id),"source_bindings":source_bindings,
    "source_sha256":binding["source_sha256"],"plan_sha256":sha(CARD/"plan.json"),"binding_sha256":sha(CARD/"run_binding.json"),
    "provider_audit":provider_audit,"native_replays":len(new_paths),"all_native_reconciliations_passed":True,
    "market_model_fits":0,"historical_stable_excess_pass":bool(gates.loc[gates.window.eq("development"),"all_pass"].all()),
    "promotion_decision":"not_promoted_reused_history","production_change":False,"new_forward_evidence":False,
    "objective_achieved":False,"model_selection":{"performed":False,"primary_candidate":"volatility_capped"},
    "benchmark":"same-cost native SOXX buy-and-hold; matched raw-target control isolates cap",
    "windows":plan["windows"],"costs_bps":plan["costs_bps"],"sidecars":list(tables)+["plan.json","run_binding.json","attempt.json","conclusion.md"],
    "artifact_policy":{"policy":"slim_card_recorder_first","workflow_config":"workflow.yaml","recorder_only_artifacts":["inputs/target_weights.pkl","portfolio_analysis/native_reports.pkl","analysis/reconciliation.pkl","analysis/independent_ledgers.pkl","analysis/bootstrap_draws.pkl"]},
    "qlib_docs_reference":["https://qlib.readthedocs.io/en/latest/component/data.html","https://qlib.readthedocs.io/en/latest/component/strategy.html","https://qlib.readthedocs.io/en/latest/component/recorder.html"],
    "limitations":plan["limitations"]}
parts=["# 策略瓶颈与风险预算优化结论", "本轮完成冻结财报预测误差归因，以及预先固定的 SOXL 波动率风险上限实验。没有训练新模型或搜索阈值。8 条新增账本均由原生 Qlib 执行，并与独立现金/股数/费用账本对账通过。", "每侧 25bp 的比较如下（累计历史已复用）："]
for window in plan["windows"]:
    parts.append(f"\n{window}：")
    for variant in ("legacy_repaired","raw_intended_weight_control","volatility_capped","soxx_buy_hold"):
        row=comparison.loc[comparison.window.eq(window)&comparison.cost_bps.eq(25)&comparison.variant.eq(variant)].iloc[0]
        parts.append(f"- {variant}：累计收益 {row.return_pct:.2f}%，最大回撤 {row.max_drawdown_pct:.2f}%，CVaR5 {row.cvar5_pct:.2f}%，相对 SOXX 财富 {row.relative_wealth_vs_soxx:.4f}。")
parts.extend([f"\n开发期稳定超额门：{summary['historical_stable_excess_pass']}。主候选固定，结果不用于选择新的上限、窗口或信号。风险下降必须结合收益损失、现金暴露和费用一起评估，不能等同于 alpha 改善。",
"财报增量误差见 forecast_mse_attribution.csv：IC 单项改善不能替代预测误差和扣费事件检验；高置信预测与实际回报偏差见 forecast_event_calibration.csv。没有按年度归因结果删年、反号或重拟合。",
"原始规则与 raw target control 的差异包含日常目标调仓语义；只有 capped vs raw control 能隔离本轮上限效果。现金空仓状态刻意保留，本方案不能修复缺少入场信号的近期空仓。年度/窗口差异见 annual_metrics.csv；未事后追加 regime 筛选。",
"不晋升，不改生产。稳定超额目标尚未完成。历史价格回溯复权、VIX 时点假设、影子信号与新账本分离、线性费用及复用历史的风险保留。下一步先取得可信新时点数据或连续前向样本；本轮不改配方继续筛选。",
f"Qlib Recorder：{recorder_id}；数据截至 {plan['as_of']}。"])
conclusion="\n\n".join(parts)+"\n"
(CARD/"conclusion.md").write_text(conclusion)
(CARD/"summary.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2,allow_nan=False)+"\n")
with R.start(experiment_name=NAME,recorder_id=recorder_id,uri=uri,resume=True):
    recorder=R.get_recorder()
    R.save_objects(artifact_path="analysis",**{"tables.pkl":tables,"bootstrap_draws.pkl":draws,"summary.pkl":summary})
    R.log_metrics(historical_stable_excess_pass=int(summary["historical_stable_excess_pass"]),native_reconciliation_pass=1)
    for name in [*tables,"summary.json","conclusion.md"]:recorder.log_artifact(str(CARD/name),artifact_path="card")
    for path in binding["source_sha256"]:recorder.log_artifact(str(ROOT/path),artifact_path="source/"+str(Path(path).parent))
print(conclusion)

[3010:MainThread](2026-10-08 10:58:04,158) INFO - qlib.workflow - [exp.py:258] - Experiment 216598613405890605 starts running ...


[3010:MainThread](2026-10-08 10:58:04,161) INFO - qlib.workflow - [recorder.py:345] - Recorder 42a5f95d21c3413bac125fa5b4f1085a starts running under Experiment 216598613405890605 ...


[3010:MainThread](2026-10-08 10:58:04,248) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


# 策略瓶颈与风险预算优化结论

本轮完成冻结财报预测误差归因，以及预先固定的 SOXL 波动率风险上限实验。没有训练新模型或搜索阈值。8 条新增账本均由原生 Qlib 执行，并与独立现金/股数/费用账本对账通过。

每侧 25bp 的比较如下（累计历史已复用）：


development：

- legacy_repaired：累计收益 342.46%，最大回撤 -70.07%，CVaR5 -7.53%，相对 SOXX 财富 1.0394。

- raw_intended_weight_control：累计收益 329.99%，最大回撤 -70.14%，CVaR5 -7.60%，相对 SOXX 财富 1.0101。

- volatility_capped：累计收益 163.40%，最大回撤 -47.23%，CVaR5 -4.49%，相对 SOXX 财富 0.6188。

- soxx_buy_hold：累计收益 325.69%，最大回撤 -45.75%，CVaR5 -4.74%，相对 SOXX 财富 1.0000。


reused_diagnostic：

- legacy_repaired：累计收益 520.55%，最大回撤 -59.73%，CVaR5 -7.10%，相对 SOXX 财富 1.2265。

- raw_intended_weight_control：累计收益 502.74%，最大回撤 -60.70%，CVaR5 -7.21%，相对 SOXX 财富 1.1913。

- volatility_capped：累计收益 230.83%，最大回撤 -44.50%，CVaR5 -4.99%，相对 SOXX 财富 0.6539。

- soxx_buy_hold：累计收益 405.97%，最大回撤 -41.36%，CVaR5 -5.40%，相对 SOXX 财富 1.0000。


recent_cash_start：

- legacy_repaired：累计收益 0.00%，最大回撤 0.00%，CVaR5 0.00%，相对 SOXX 财富 0.8611。

- raw_intended_weight_control：累计收益 0.00%，最大回撤 0.00%，CVaR5 0.00%，相对 SOXX 财富 0.8611。

- volatility_

# 策略瓶颈与风险预算优化结论

本轮完成冻结财报预测误差归因，以及预先固定的 SOXL 波动率风险上限实验。没有训练新模型或搜索阈值。8 条新增账本均由原生 Qlib 执行，并与独立现金/股数/费用账本对账通过。

每侧 25bp 的比较如下（累计历史已复用）：


development：

- legacy_repaired：累计收益 342.46%，最大回撤 -70.07%，CVaR5 -7.53%，相对 SOXX 财富 1.0394。

- raw_intended_weight_control：累计收益 329.99%，最大回撤 -70.14%，CVaR5 -7.60%，相对 SOXX 财富 1.0101。

- volatility_capped：累计收益 163.40%，最大回撤 -47.23%，CVaR5 -4.49%，相对 SOXX 财富 0.6188。

- soxx_buy_hold：累计收益 325.69%，最大回撤 -45.75%，CVaR5 -4.74%，相对 SOXX 财富 1.0000。


reused_diagnostic：

- legacy_repaired：累计收益 520.55%，最大回撤 -59.73%，CVaR5 -7.10%，相对 SOXX 财富 1.2265。

- raw_intended_weight_control：累计收益 502.74%，最大回撤 -60.70%，CVaR5 -7.21%，相对 SOXX 财富 1.1913。

- volatility_capped：累计收益 230.83%，最大回撤 -44.50%，CVaR5 -4.99%，相对 SOXX 财富 0.6539。

- soxx_buy_hold：累计收益 405.97%，最大回撤 -41.36%，CVaR5 -5.40%，相对 SOXX 财富 1.0000。


recent_cash_start：

- legacy_repaired：累计收益 0.00%，最大回撤 0.00%，CVaR5 0.00%，相对 SOXX 财富 0.8611。

- raw_intended_weight_control：累计收益 0.00%，最大回撤 0.00%，CVaR5 0.00%，相对 SOXX 财富 0.8611。

- volatility_capped：累计收益 0.00%，最大回撤 0.00%，CVaR5 0.00%，相对 SOXX 财富 0.8611。

- soxx_buy_hold：累计收益 16.13%，最大回撤 -10.99%，CVaR5 -5.28%，相对 SOXX 财富 1.0000。


开发期稳定超额门：False。主候选固定，结果不用于选择新的上限、窗口或信号。风险下降必须结合收益损失、现金暴露和费用一起评估，不能等同于 alpha 改善。

财报增量误差见 forecast_mse_attribution.csv：IC 单项改善不能替代预测误差和扣费事件检验；高置信预测与实际回报偏差见 forecast_event_calibration.csv。没有按年度归因结果删年、反号或重拟合。

原始规则与 raw target control 的差异包含日常目标调仓语义；只有 capped vs raw control 能隔离本轮上限效果。现金空仓状态刻意保留，本方案不能修复缺少入场信号的近期空仓。年度/窗口差异见 annual_metrics.csv；未事后追加 regime 筛选。

不晋升，不改生产。稳定超额目标尚未完成。历史价格回溯复权、VIX 时点假设、影子信号与新账本分离、线性费用及复用历史的风险保留。下一步先取得可信新时点数据或连续前向样本；本轮不改配方继续筛选。

Qlib Recorder：42a5f95d21c3413bac125fa5b4f1085a；数据截至 2026-09-25。

## 瓶颈归因与改进代价

财报增量没有修复原模型的误差：cash5 / leverage5 的增量预测与 base 残差相关均约 -0.078；2022 年分别贡献整段 SSE 恶化的 87.22% / 83.30%。这是对同一完整历史的描述，不支持删年份或反转信号。25bp 的加财报事件：cash5 238 次预测平均净 +39.90bp、实际 -45.39bp；leverage5 424 次预测 +201.63bp、实际 -186.88bp。此处 full-minus-base 包含联合重拟合的基础系数变化，不是对单个财报特征的因果估计。

风险上限相对相同调仓方式的 raw control 将开发期回撤改善 22.91 个百分点，但年化收益由 23.20% 降至 14.86%，SOXX 为 23.02%。候选的 CVaR 较 SOXX 好，最大回撤仍更差；七年全部负超额。不能据此证明已产生独立于杠杆的择时 alpha。旧规则改为每日目标调仓后换手本身由约 5.16 倍增加到 15.01 倍，候选约 14.66 倍；只减少最大仓位不会自动减少交易成本。

已排除一个直接修复假设：单独加波动率上限不能同时保留原收益并满足 SOXX 风险门。稳定超额、有效入场与收益预测仍未解决；不应把本轮风险缩减候选替换到生产。

## 验证与归档

126 项相关测试通过；独立审查核验 8 条完整路径、9 项源码身份、12 个因果目标点及全部已登记区间。账户/现金/股数最大误差为 0，费用误差约 3.50e-12。默认全量 pytest 在改动前后均被 15 个历史归档测试同名导入冲突阻断；旧 Qlib 卡片套件在改动前已有 61 项缺失文件失败，不能报告全仓通过。

执行源码与 plan/run_binding 保持原封存；最终文字结论在执行后依据保存的小表撰写并同步 notebook，未再回放或选择其他方案。新 Recorder 的 artifact URI 仅迁移到主仓，映射见 relocation.json；完整逐文件来源/用途/范围/hash 见 verification_manifest.json。
